In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# Define State
class EmployeeState(TypedDict):
    employee_name: str
    monthly_salary: int
    working_days: int
    completed_projects: int
    
    yearly_salary: int
    bonus_amount: int
    project_status: str
    summary: str
    

# Define Nodes (Sequential pipeline)
def calculate_yearly_salary(state: EmployeeState):
    yearly_salary = state['monthly_salary'] * 12
    return {'yearly_salary': yearly_salary}


def calculate_bonus(state: EmployeeState):
    bonus_amount = state['yearly_salary'] * 2
    return {'bonus_amount': bonus_amount}


def project_evaluation(state: EmployeeState):
    if state['completed_projects'] >= 5:
        status = 'Excellent'
    else:
        status = 'Average'
    return {'project_status': status}


def summary(state: EmployeeState):
    summary_text = (
        f"Employee {state['employee_name']} has a yearly salary of "
        f"{state['yearly_salary']}, a bonus of {state['bonus_amount']}, "
        f"and a project status of {state['project_status']}."
    )
    return {'summary': summary_text}


# Create graph
graph = StateGraph(EmployeeState)

# Add nodes
graph.add_node('calculate_yearly_salary', calculate_yearly_salary)
graph.add_node('calculate_bonus', calculate_bonus)
graph.add_node('project_evaluation', project_evaluation)
graph.add_node('summary', summary)

# Add Edges (Linear sequential chain ensures all keys are safely in state before summary runs)
graph.add_edge(START, 'calculate_yearly_salary')
graph.add_edge('calculate_yearly_salary', 'calculate_bonus')
graph.add_edge('calculate_bonus', 'project_evaluation')
graph.add_edge('project_evaluation', 'summary')
graph.add_edge('summary', END)

workflow = graph.compile()

# Initial State
initial_state = {
    'employee_name': 'Mbr Sagor',
    'monthly_salary': 120000,
    'working_days': 26,
    'completed_projects': 7
}

result = workflow.invoke(initial_state)
print(result)

{'employee_name': 'Mbr Sagor', 'monthly_salary': 120000, 'working_days': 26, 'completed_projects': 7, 'yearly_salary': 1440000, 'bonus_amount': 2880000, 'project_status': 'Excellent', 'summary': 'Employee Mbr Sagor has a yearly salary of 1440000, a bonus of 2880000, and a project status of Excellent.'}
